# 14. Fitting

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

Fitting is the forward pass run backwards. Forward is "parameters → points". Fitting is "the points have been measured; find parameters whose forward points lie close to them".

## One unknown on the toy

The toy finger's DIP vertex is produced by an unknown PIP flexion. The next cell sets the true angle to 40 degrees and minimizes only the DIP-vertex error, searching from -20 degrees to 120 degrees.


In [ ]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

import toy_mano as toy
from scipy.optimize import minimize_scalar

secret_deg = 40.0
pose = np.zeros((4, 3))
pose[2, 0] = np.deg2rad(secret_deg)
target = toy.forward(pose)["vertices"][3]

def loss(degree):
    trial = np.zeros((4, 3))
    trial[2, 0] = np.deg2rad(degree)
    tip = toy.forward(trial)["vertices"][3]
    return float(np.sum((tip - target) ** 2))

found = minimize_scalar(loss, bounds=(-20.0, 120.0), method="bounded")
print(f"recovered {found.x:.2f} degrees, true angle {secret_deg:.2f} degrees")
assert abs(found.x - secret_deg) < 0.05


recovered 40.00 degrees, true angle 40.00 degrees


The loss is

```text
loss(angle) = || forward(angle) DIP vertex - target ||²
```

## The same loop on the real model

There are more unknowns:

| Quantity | Count | A usual constraint |
|---|---|---|
| `global_orient` | 3 | An axis-angle, usually not several full turns |
| Fingers | 45 axis-angles, or `n` PCA coefficients | A short PCA vector makes gestures smoother, and cannot hit an arbitrary angle |
| `betas` | 10 | Penalize the absolute value so the hand does not grow or shrink without bound |
| `transl` | 3 | Same coordinate frame as the target, in metres |

When the target is a joint, compare it with `out["joints"]`. When the target is a fingertip, compare it with `out["tips"]`. The thumb tip is vertex 744, not joint 15.

A minimal loss is

```text
loss = data term + regularizer
data term = joint error + fingertip error
regularizer = sum of squares of betas + sum of squares of PCA coefficients
```

The optimizer calls `forward` again and again and steps the parameters downhill. The next cell searches only the index PIP angle on the official right hand. The target is the index tip produced by that angle.


In [ ]:
from scipy.optimize import minimize_scalar
from official_mano import require_model

model = require_model()
secret = 55.0
hand_pose = np.zeros(45)
hand_pose[5] = np.deg2rad(secret)
target = model.forward(hand_pose=hand_pose, flat_hand=True)["tips"]["index"]

def loss(degree):
    trial = np.zeros(45)
    trial[5] = np.deg2rad(degree)
    tip = model.forward(hand_pose=trial, flat_hand=True)["tips"]["index"]
    return float(np.sum((tip - target) ** 2))

found = minimize_scalar(loss, bounds=(0.0, 90.0), method="bounded")
print(f"recovered index PIP {found.x:.2f} degrees, true {secret:.2f} degrees")
assert abs(found.x - secret) < 0.05


recovered index PIP 55.00 degrees, true 55.00 degrees


The coordinate frames have to match first. MANO's points are in the model frame, in metres. Points in a camera are often in millimetres. Put both sides on the same origin, the same axes, and the same unit, and only then compute the error. The datasets in chapter 15 also need their joint order rearranged first.

## Check

When the target is the thumb tip, the loss subtracts `tips["thumb"]`, vertex 744. `joints[15]` is the thumb IP. On the flat template it sits about 35 mm away from that tip.
